# Munkhpurev Char-GPT Training

This notebook trains a small decoder-only transformer **from scratch** —
character by character, no pretrained weights — on Munkhpurev's own
one-on-one Messenger/Instagram history, so the trained model completes text
the way *he* texts. It follows the same approach as Karpathy's "Let's build
GPT" walkthrough (Vaswani et al., *Attention Is All You Need*,
arXiv:1706.03762; Brown et al., GPT-3, arXiv:2005.14165), scaled down to a
personal-chat-sized dataset instead of internet-scale text.

The dataset (`munkhpurev_training_data.json`) was built locally from several
years of exported chat threads: cleaned of system/junk messages and
business-bot conversations, transliterated to a single Latin script, and
merged so every conversation strictly zigzags `Other / Munkhpurev / Other /
Munkhpurev / ...` with no consecutive same-speaker lines. That preprocessing
already happened before this notebook — here we only train on the result.

**Gotchas:**
- **Turn on the GPU accelerator** (Settings → Accelerator → GPU) before
  running. The notebook auto-detects CUDA and will happily run on CPU
  instead if it's off — just 10-50x slower for a model this size, with no
  warning beyond the printed `device: cpu`.
- **Stage 3 searches `/kaggle/input` recursively** for
  `munkhpurev_training_data.json`. If your uploaded dataset has a different
  file name, either rename it to match or edit `DATA_FILENAME` in Stage 2.
- **Stage 7 (the training loop) is resumable.** Re-running that cell
  continues from the last saved iteration in `checkpoint.pt` rather than
  starting over — useful if a Kaggle session times out mid-run. If you want
  a clean restart instead, run the "delete checkpoint" cell at the top of
  Stage 7 first.
- **Training windows never cross a conversation boundary.** A short
  conversation (fewer characters than `BLOCK_SIZE`) gets cyclically
  repeated to fill its own training window rather than reaching into a
  different, unrelated conversation — see Stage 5 for why.

**Stages:** 1 setup · 2 config · 3 load dataset · 4 model architecture ·
5 data prep & batching · 6 build model + sanity check · 7 training loop
(resumable) · 8 chat with the trained model · 9 save & summarize.

## Stage 1 · Setup

Kaggle's GPU notebooks ship with PyTorch pre-installed, so the common case
here is "nothing to install." But checking with `importlib.util.find_spec`
instead of assuming it's present means this same notebook still works if
run somewhere else (a fresh Colab runtime, a local Jupyter server) — it
installs `torch` only if it's actually missing, rather than either crashing
on import or blindly re-installing/upgrading a GPU build that's already
correctly configured for this machine.

This cell also prints whether CUDA is visible *right now*, before any real
work happens — if it prints `cuda: False` here, stop and fix the
accelerator setting rather than finding out after a slow training run.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)

import torch

print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device name: {torch.cuda.get_device_name(0)}")

## Stage 2 · Config

Every tunable constant lives here, in one place, as UPPER_CASE
module-level variables — so a re-run with different settings means editing
this cell only, not hunting through the model or training-loop code below.

Sizing note: `N_EMBD=384, N_LAYER=6` (~11M parameters) targets roughly the
same parameters-per-training-character ratio as nanoGPT's own
`train_shakespeare_char.py` reference config (~10 params/char on 1.1M
characters of Shakespeare) applied to this dataset's ~1.9M training
characters. That ratio was chosen for a GPU, not a laptop CPU — on CPU this
config would take hours just to reach a few hundred iterations.

`BATCH_SIZE=128` (doubled from a single-GPU 64) matters specifically for
the 2-GPU setup in Stage 6: `nn.DataParallel` splits each batch evenly
across the visible GPUs before the forward pass, so a batch of 64 would
mean each T4 only ever sees 32 examples per step — leaving the second GPU
underused even though it's "on". Doubling the batch keeps each GPU's
per-step workload the same as the single-GPU config did.

In [ ]:
DATA_FILENAME = "munkhpurev_training_data.json"
CHECKPOINT_PATH = "checkpoint.pt"

BLOCK_SIZE = 256       # characters of context per prediction
BATCH_SIZE = 128       # split across GPUs by DataParallel in Stage 6
N_EMBD = 384
N_HEAD = 6
N_LAYER = 6
DROPOUT = 0.2
LEARNING_RATE = 1e-3
MAX_ITERS = 5000
EVAL_INTERVAL = 250
EVAL_ITERS = 100
TRAIN_FRAC = 0.9        # fraction of conversations used for training vs val
SEED = 1337

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
print(f"device: {DEVICE}")

## Stage 3 · Locate and load the dataset

The dataset was uploaded as a Kaggle Dataset, which mounts read-only under
`/kaggle/input/<dataset-slug>/...` — the exact slug depends on what you
named it at upload time, so rather than hardcoding a guessed path, this
cell searches recursively for `DATA_FILENAME` under `/kaggle/input`. If
that search comes up empty (e.g. running this outside Kaggle), it falls
back to looking in the current directory, so the same notebook also works
against a local copy of the file.

The JSON is a list of conversations, each `{"turns": [{"speaker", "text"},
...]}`, already zigzag-merged and cleaned. We reconstruct each conversation
back into the flat `"Speaker: text"` line format the character-level model
actually trains on — the JSON structure only matters for keeping
conversation boundaries intact in Stage 5, not for what the model sees
character-by-character.

In [ ]:
import json
from pathlib import Path

candidates = list(Path("/kaggle/input").rglob(DATA_FILENAME)) if Path("/kaggle/input").exists() else []
if not candidates:
    local = Path(DATA_FILENAME)
    if local.exists():
        candidates = [local]

if not candidates:
    raise FileNotFoundError(
        f"Couldn't find {DATA_FILENAME} under /kaggle/input or the current directory. "
        "Check the dataset was attached to this notebook, or edit DATA_FILENAME in Stage 2."
    )

data_path = candidates[0]
print(f"using dataset: {data_path}")

with open(data_path, "r", encoding="utf-8") as f:
    conversations = json.load(f)


def conv_to_text(conv):
    return "\n".join(f"{t['speaker']}: {t['text']}" for t in conv["turns"])


conv_texts = [conv_to_text(c) for c in conversations]
print(f"{len(conv_texts)} conversations, {sum(len(t) for t in conv_texts):,} total characters")

## Stage 4 · Model architecture

A small decoder-only transformer, teaching-scale version of the GPT
architecture: token + positional embeddings feed a stack of pre-norm
transformer blocks (causal self-attention + a 4x-widening MLP), ending in a
final layernorm and a linear head back to vocabulary size. This is one
cell rather than several because the pieces (`Head`, `MultiHeadAttention`,
`FeedForward`, `Block`, `GPT`) only make sense assembled together and none
of them do anything demonstrable on their own — splitting them across
cells would just force scrolling back and forth to see how they connect.

`generate()` is the only piece not used during training — it's what
Stage 8 calls to sample new text from the trained model, one character at
a time, feeding each sampled character back in as the next input.

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

# ============ Causal self-attention (one head) ============
class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v


# ============ Multiple heads run in parallel, concatenated ============
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList(
            [Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)]
        )
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


# ============ Position-wise feed-forward (the "thinking" step) ============
class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


# ============ One transformer block: attend, then think, both with residuals ============
class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


# ============ Full model: embed, stack of blocks, project to vocab ============
class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


print("model classes defined.")

## Stage 5 · Vocabulary, train/val split, and conversation-aware batching

Three design decisions here, each because the naive version would be
subtly wrong for this dataset:

1. **Vocabulary is every unique character across all conversations**
   (char-level, not a subword tokenizer) — matches the "from scratch, no
   pretrained tokenizer" approach this whole notebook follows.
2. **The train/val split is by conversation, not by raw character
   position.** Splitting by character position (e.g. "first 90% of the
   concatenated text") would put the *end* of a conversation in train and
   its *start* in val, or vice versa — leaking the same conversation across
   both sets. Splitting by whole conversation avoids that.
3. **`get_batch` samples a random window from a randomly chosen
   conversation, weighted by that conversation's length** (so a 50,000-char
   conversation is proportionally more likely to be sampled from than a
   200-char one — matching what uniform-character sampling over the old
   flat-file approach used to do), **and a window never spans two
   conversations.** A conversation shorter than `BLOCK_SIZE` characters
   gets cyclically repeated (`conv.repeat(...)`) to fill its own window
   rather than the window silently bleeding into a different, unrelated
   conversation the way naive whole-corpus concatenation would allow.

In [ ]:
import random

chars = sorted(set("\n".join(conv_texts)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}


def encode(s):
    return [stoi[c] for c in s]


def decode(ids):
    return "".join(itos[i] for i in ids)


split_rng = random.Random(SEED)
order = list(range(len(conv_texts)))
split_rng.shuffle(order)
n_train_convs = int(TRAIN_FRAC * len(order))
train_idx, val_idx = order[:n_train_convs], order[n_train_convs:]

train_convs = [torch.tensor(encode(conv_texts[i]), dtype=torch.long) for i in train_idx]
val_convs = [torch.tensor(encode(conv_texts[i]), dtype=torch.long) for i in val_idx]
train_weights = [len(c) for c in train_convs]
val_weights = [len(c) for c in val_convs]
train_char_count = sum(train_weights)
val_char_count = sum(val_weights)

print(f"vocab_size: {vocab_size}")
print(f"train: {len(train_convs)} conversations, {train_char_count:,} chars")
print(f"val:   {len(val_convs)} conversations, {val_char_count:,} chars")


def get_batch(split):
    convs, weights = (train_convs, train_weights) if split == "train" else (val_convs, val_weights)
    picks = random.choices(convs, weights=weights, k=BATCH_SIZE)
    xs, ys = [], []
    for conv in picks:
        if len(conv) < BLOCK_SIZE + 1:
            reps = BLOCK_SIZE // len(conv) + 2
            conv = conv.repeat(reps)
        i = random.randint(0, len(conv) - BLOCK_SIZE - 1)
        xs.append(conv[i:i + BLOCK_SIZE])
        ys.append(conv[i + 1:i + BLOCK_SIZE + 1])
    x = torch.stack(xs).to(DEVICE)
    y = torch.stack(ys).to(DEVICE)
    return x, y


@torch.no_grad()
def estimate_loss(model):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            x, y = get_batch(split)
            _, loss = model(x, y)
            # under nn.DataParallel, loss comes back as one value PER GPU
            # (each replica computes its own scalar); .mean() collapses
            # that to a single number. A no-op on a single GPU, where loss
            # is already a 0-dim scalar tensor.
            losses[k] = loss.mean().item()
        out[split] = losses.mean().item()
    model.train()
    return out

## Stage 6 · Build the model (persistent — run once)

The model and optimizer are created **once, here**, not inside the
training loop — Stage 7 reuses these same `model`/`optimizer` objects on
every re-run, so re-running Stage 7 after an interruption continues
training the same weights instead of starting from a fresh random
initialization. Re-running *this* cell, by contrast, throws away any
in-progress training and starts over (unless a checkpoint is found on
disk, in which case it loads those weights back in).

**Multi-GPU (`nn.DataParallel`):** if more than one GPU is visible, the
model gets wrapped so each training step splits its batch across all of
them automatically. Three things fall out of that wrapping and are worth
knowing about before they show up as confusing errors elsewhere:

- `model(x, y)` still works for the forward/backward pass (DataParallel
  forwards `__call__`), but custom methods like `.generate()` are **not**
  forwarded — only `forward()` is. So Stage 8 calls `.generate()` on
  `raw_model` (the unwrapped module we keep a reference to below), not on
  `model` directly.
- Checkpoints are saved/loaded from `raw_model.state_dict()`, not
  `model.state_dict()` — a DataParallel-wrapped state dict has `module.`
  prefixed onto every key, which would silently break loading the same
  checkpoint later on a single GPU or CPU (e.g. locally in `chat.py`).
  Saving the unwrapped state dict keeps the checkpoint format identical
  either way.
- Each GPU replica computes its own scalar loss, so `model(x, y)` returns
  a loss *tensor* with one value per GPU under DataParallel, not a single
  scalar — see the `.mean()` calls added in Stage 5's `estimate_loss` and
  in Stage 7's training loop.

The sanity check on a fresh (no-checkpoint) start is Karpathy's classic
one: a randomly-initialized model predicting uniformly across `vocab_size`
classes should have a loss near `-ln(1/vocab_size)`. If the printed
"initial loss" is wildly different from "expected," something about the
data or model setup is broken before you've spent any real training time
finding out.

In [ ]:
model = GPT(vocab_size, N_EMBD, N_HEAD, N_LAYER, BLOCK_SIZE, DROPOUT).to(DEVICE)

n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    print(f"{n_gpus} GPUs visible — wrapping model in nn.DataParallel")
    model = nn.DataParallel(model)

# the unwrapped module: used for checkpoint save/load and for calling
# .generate() in Stage 8, both of which DataParallel doesn't handle for us
raw_model = model.module if isinstance(model, nn.DataParallel) else model

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
start_iter = 0

try:
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    raw_model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_iter = ckpt["iter"] + 1
    print(f"resumed from {CHECKPOINT_PATH} at iter {start_iter}")
except FileNotFoundError:
    import math
    expected = -math.log(1 / vocab_size)
    x, y = get_batch("train")
    _, loss = model(x, y)
    print(f"initial loss: {loss.mean().item():.4f} (expected ~{expected:.4f} if untrained)")

n_params = sum(p.numel() for p in raw_model.parameters())
print(f"model params: {n_params:,}")
print(f"start_iter: {start_iter} / {MAX_ITERS}")

## Stage 7 · Training loop (resumable — run as often as needed)

Checkpoints every `EVAL_INTERVAL` steps to `CHECKPOINT_PATH`, so an
interrupted Kaggle session loses at most `EVAL_INTERVAL` steps of progress,
not the whole run. Re-running this cell picks up from `start_iter` (set in
Stage 6) rather than restarting — that's *why* Stage 6 is separate: the
model/optimizer state has to survive across repeated calls to this cell
for resuming to work at all.

If you want a clean restart instead of resuming, run the delete-checkpoint
cell below, then re-run Stage 6 (to build a fresh model) before running
this cell again.

In [ ]:
# Run this cell only if you want to discard checkpoint.pt and start over.
import os
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print(f"deleted {CHECKPOINT_PATH} — re-run Stage 6 to build a fresh model before training.")
else:
    print("no checkpoint to delete.")

In [ ]:
for it in range(start_iter, MAX_ITERS):
    if it % EVAL_INTERVAL == 0 or it == MAX_ITERS - 1:
        losses = estimate_loss(model)
        print(f"step {it}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        torch.save({
            "model": raw_model.state_dict(),  # unwrapped: no 'module.' prefix
            "optimizer": optimizer.state_dict(),
            "iter": it,
            "vocab": chars,
            "config": dict(n_embd=N_EMBD, n_head=N_HEAD, n_layer=N_LAYER,
                            block_size=BLOCK_SIZE, dropout=DROPOUT),
        }, CHECKPOINT_PATH)

    x, y = get_batch("train")
    _, loss = model(x, y)
    loss = loss.mean()  # one value per GPU under DataParallel -> single scalar
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

start_iter = MAX_ITERS
print("done training.")

## Stage 8 · Chat with the trained model

`model` is already trained and in memory from Stage 7 — this stage doesn't
reload anything, it just switches it to eval mode and defines `talk()`,
which stays reusable across as many calls as you like. `HISTORY`
accumulates across calls in the same running conversation, the same way
`chat.py` does locally: each call appends both your line and the model's
reply, so later replies see earlier context (up to `BLOCK_SIZE`
characters back).

Generation calls `raw_model.generate(...)`, not `model.generate(...)` —
`.generate()` is a custom method on the `GPT` class, and `nn.DataParallel`
only forwards `forward()`/`__call__`, so calling it on the wrapped `model`
would raise `AttributeError` whenever 2 GPUs are active. `raw_model` (set
in Stage 6) is always the plain, unwrapped module either way, so this line
works the same whether training used 1 GPU or 2.

The model is forced to speak as `Munkhpurev` by ending the prompt with
that name prefix before sampling — otherwise it would be free to predict
*either* speaker next, since both appear throughout training. Only the
first generated line is kept as the reply, since generation can run past
one turn if not stopped.

In [ ]:
raw_model.eval()
HISTORY = ""


def talk(their_line, max_new_tokens=200, temperature=0.8, top_k=40):
    global HISTORY
    HISTORY += f"Other: {their_line}\n"

    prefix = "Munkhpurev: "
    room = BLOCK_SIZE - len(prefix)
    context = HISTORY[-room:] if room > 0 else ""
    prompt = context + prefix

    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    out = raw_model.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    generated = decode(out[0].tolist())[len(prompt):]
    reply = generated.split("\n")[0].strip() or "..."

    HISTORY += f"{prefix}{reply}\n"
    print(f"Munkhpurev> {reply}")
    return reply


# try it out — call talk() again as many times as you like, it keeps the running HISTORY
talk("hey what are you up to")
talk("haha nice, miss you")

## Stage 9 · Save & summarize

`checkpoint.pt` is already saved incrementally by Stage 7 — this stage
just confirms it exists, reports what's in it, and prints a copy-pasteable
summary so you know what to expect after downloading it from Kaggle's
notebook output and dropping it back into the local project folder for
`chat.py` to use.

In [ ]:
import os

assert os.path.exists(CHECKPOINT_PATH), "no checkpoint found — did Stage 7 finish at least one eval_interval?"

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
size_mb = os.path.getsize(CHECKPOINT_PATH) / (1024 * 1024)

print("=== training summary ===")
print(f"checkpoint: {CHECKPOINT_PATH} ({size_mb:.1f} MB)")
print(f"trained to iteration: {ckpt['iter']} / {MAX_ITERS}")
print(f"model params: {n_params:,}")
print(f"vocab size: {len(ckpt['vocab'])}")
print(f"trained on: {train_char_count:,} chars across {len(train_convs)} conversations")
print()
print("download checkpoint.pt from this notebook's output files, then locally:")
print("  cp <downloaded checkpoint.pt> /home/lunatic/Desktop/ml/conversation_dataset/checkpoint.pt")
print("  python chat.py")